# FlakeBench fine-tuning on Kaggle

Fine-tunes the FlakyLens CodeBERT classifier on `FlakeBench/FlakeBench_dataset.csv` inside a Kaggle
notebook, using the repository's own training script (`src/Bert_train_per_project.py`) without
changing its logic.

This notebook is configured as a **smoke run** by default: one project group, 3 epochs, roughly
10-15 minutes on a T4. Change the settings in the Configuration cell to scale up to the full
reproduction (4 groups, 40 epochs, ~8-10 hours).

**Before running**

1. Notebook settings: Accelerator = **GPU T4 x2**. The training code hardcodes
   `torch.device("cuda")`, so it fails immediately without a GPU.

   Do **not** pick P100. It is compute capability sm_60, and Kaggle's current PyTorch build only
   ships kernels for sm_70 and above, so every kernel launch fails with `no kernel image is
   available for execution on the device`. T4 is sm_75 and works. Only one GPU is used - the
   training code has no multi-GPU support - so the second T4 stays idle.
2. Notebook settings: Internet = **ON** (needed for pip and for downloading `microsoft/codebert-base`).
3. Make the repository available to the notebook. The default is a clone of the upstream artifact
   repository:

   ```
   git clone https://github.com/UT-SE-Research/FlakyLens.git /kaggle/working/FlakyLens
   ```

   Point `GITHUB_URL` at your own fork instead if you have local changes, or switch `REPO_SOURCE`
   to `"kaggle_dataset"` to work from an uploaded Kaggle Dataset.

## 1. Configuration

In [ ]:
# Where the repository comes from:
#   "github"         - clone a public/authenticated GitHub URL (needs Internet ON)
#   "kaggle_dataset" - copy from an attached Kaggle Dataset (/kaggle/input/...) into the working dir
#   "existing"       - the repo is already at REPO_DIR
REPO_SOURCE = "github"

GITHUB_URL   = "https://github.com/UT-SE-Research/FlakyLens.git"  # used when REPO_SOURCE == "github"
KAGGLE_INPUT = "/kaggle/input/flakylens"                        # used when REPO_SOURCE == "kaggle_dataset"
REPO_DIR     = "/kaggle/working/FlakyLens"

# Dataset, relative to REPO_DIR
DATASET_REL = "FlakeBench/FlakeBench_dataset.csv"

# Training scale.
#   GROUPS = 1, EPOCHS = 3   -> smoke run, ~10-15 min on T4
#   GROUPS = 4, EPOCHS = 40  -> full reproduction, ~8-10 h (fits one 12 h session)
GROUPS     = 1
EPOCHS     = 3
BATCH_SIZE = 8      # 8 is the repository default; raise to 16 only if GPU memory allows

# Argument values expected by the repository scripts
DATA_DIR_NAME      = "FlakyLens_Categorization_PerProject-Data"
TECHNIQUE          = "BERT-FlakeBench"
WEIGHTS_PREFIX_REL = "../models/per_project_model_weights_on__dataset"

SRC = f"{REPO_DIR}/src"
print("repo:", REPO_DIR, "| groups:", GROUPS, "| epochs:", EPOCHS, "| batch:", BATCH_SIZE)

## 2. Environment check

Confirms a GPU is actually attached before anything expensive runs.

In [ ]:
import torch, sys, platform

print("python  :", sys.version.split()[0], "on", platform.platform())
print("torch   :", torch.__version__)
print("cuda    :", torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print("device  :", torch.cuda.get_device_name(0), f"(sm_{major}{minor})")
    print("memory  : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    print("kernels :", " ".join(torch.cuda.get_arch_list()))

    # Kaggle's PyTorch build dropped sm_60, so a P100 accepts the allocation but fails on the first
    # kernel launch with "no kernel image is available for execution on the device". Catch it here.
    if f"sm_{major}{minor}" not in torch.cuda.get_arch_list():
        raise SystemExit(
            f"This PyTorch build has no kernels for sm_{major}{minor} "
            f"({torch.cuda.get_device_name(0)}). Switch the accelerator to GPU T4 x2 and restart.")

    # Prove it end to end rather than trusting the arch list.
    (torch.randn(64, 64, device="cuda") @ torch.randn(64, 64, device="cuda")).sum().item()
    print("matmul  : ok")
else:
    raise SystemExit("No GPU. Enable the GPU accelerator in the notebook settings and restart.")

## 3. Get the repository into the working directory

The training script writes split CSVs next to its own source files, so the repository must live on a
writable path. `/kaggle/input` is read-only, which is why the Kaggle Dataset option copies rather
than reads in place.

In [ ]:
import os, shutil, subprocess

# Equivalent one-liner, if you prefer to run it by hand:
#   !git clone https://github.com/UT-SE-Research/FlakyLens.git /kaggle/working/FlakyLens

if REPO_SOURCE == "github":
    if not os.path.exists(REPO_DIR):
        print("cloning", GITHUB_URL)
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)
    else:
        print("already present:", REPO_DIR)

elif REPO_SOURCE == "kaggle_dataset":
    if not os.path.exists(REPO_DIR):
        shutil.copytree(KAGGLE_INPUT, REPO_DIR)
    else:
        print("already present:", REPO_DIR)

elif REPO_SOURCE == "existing":
    assert os.path.isdir(REPO_DIR), f"{REPO_DIR} does not exist"

else:
    raise ValueError(f"unknown REPO_SOURCE: {REPO_SOURCE}")

os.makedirs(f"{REPO_DIR}/models", exist_ok=True)
os.makedirs(f"{REPO_DIR}/results", exist_ok=True)

assert os.path.isfile(f"{SRC}/Bert_train_per_project.py"), "src/Bert_train_per_project.py not found"
assert os.path.isfile(f"{REPO_DIR}/{DATASET_REL}"), f"{DATASET_REL} not found"
print("repository ready")

## 4. Install dependencies

**Do not** run `pip install -r requirements.txt` here. It pins `torch>=2.3,<2.4` and `numpy==1.23.5`,
which replaces Kaggle's CUDA-matched torch build and breaks GPU support.

`transformers` is pinned to 4.40.1 on purpose: the training code uses `from transformers import
AdamW` and the `pad_to_max_length=True` tokenizer argument, both of which are gone in later releases.

In [ ]:
!pip install -q transformers==4.40.1 captum==0.7.0 imbalanced-learn javalang tensorboard interpret==0.6.1 selenium

In [ ]:
# Verify the two version-sensitive imports before spending GPU time on them.
import transformers
print("transformers:", transformers.__version__)

try:
    from transformers import AdamW
    print("transformers.AdamW available")
except ImportError:
    print("transformers.AdamW missing - patching the training script to use torch.optim.AdamW")
    import pathlib
    p = pathlib.Path(f"{SRC}/Bert_train_per_project.py")
    s = p.read_text(encoding="utf-8")
    s = s.replace("from transformers import AdamW, AutoTokenizer, AutoModel, AutoConfig",
                  "from torch.optim import AdamW\nfrom transformers import AutoTokenizer, AutoModel, AutoConfig")
    p.write_text(s, encoding="utf-8")

## 5. Dataset sanity check

Confirms the column names the training script expects. Note that in this dataset `category` holds the
numeric label 0-5 used for training and `label` holds the human-readable category name, which is the
opposite of what the README describes.

In [ ]:
import pandas as pd

df = pd.read_csv(f"{REPO_DIR}/{DATASET_REL}")
print("rows:", len(df), "| projects:", df["project"].nunique())
print("columns:", list(df.columns))
print()
print("training label ('category'):")
print(df["category"].value_counts().sort_index().to_string())
print()
print("category names ('label'):")
print(df["label"].value_counts().to_string())

assert {"full_code", "category", "project"} <= set(df.columns)
assert set(df["category"].unique()) <= set(range(6)), "'category' must be numeric 0-5"

## 6. Patch the scripts for the chosen scale

Three edits, all idempotent - re-running this cell will not stack them:

- number of epochs (`epochs = 40` in the training script),
- batch size,
- number of project groups actually run, so a smoke run trains and evaluates one group instead of
  four.

Everything else - the model, the loss function, the split logic - is left untouched.

In [ ]:
import re, pathlib

GUARD = "# kaggle-run: group limit"

def limit_groups(src, n):
    if GUARD in src:
        return re.sub(r"train_files, test_files = train_files\[:\d+\], test_files\[:\d+\]",
                      f"train_files, test_files = train_files[:{n}], test_files[:{n}]", src)
    def repl(m):
        nl, indent, stmt = m.group(1), m.group(2), m.group(3)
        return (f"{nl}{indent}{GUARD}"
                f"{nl}{indent}train_files, test_files = train_files[:{n}], test_files[:{n}]"
                f"{nl}{indent}{stmt}")
    return re.sub(r"(\n)([ \t]*)(assert len\(train_files\) == len\(test_files\))", repl, src, count=1)

# training script
p = pathlib.Path(f"{SRC}/Bert_train_per_project.py")
s = p.read_text(encoding="utf-8")
s = re.sub(r"epochs = \d+", f"epochs = {EPOCHS}", s, count=1)
s = re.sub(r"batch_size = \d+(\s*# Define the batch size)", f"batch_size = {BATCH_SIZE}\\1", s, count=1)
s = limit_groups(s, GROUPS)
p.write_text(s, encoding="utf-8")

# evaluation script: it must evaluate the same groups that were trained, otherwise it fails
# loading checkpoints that do not exist
p = pathlib.Path(f"{SRC}/Testing_per_project.py")
s = p.read_text(encoding="utf-8")
s = limit_groups(s, GROUPS)
p.write_text(s, encoding="utf-8")

# show the effect of the patches
s = pathlib.Path(f"{SRC}/Bert_train_per_project.py").read_text(encoding="utf-8")
for line in s.splitlines():
    if re.search(r"^\s*(epochs = |batch_size = |train_files, test_files = )", line):
        print(line.strip())

## 7. Train

Arguments, in order: dataset CSV, checkpoint path prefix, split output directory, technique tag.

The repository's `per_project_prediction.sh` is deliberately bypassed - it hardcodes a FlakyLens
dataset path that does not exist in this repository.

The script first builds per-project train/test groups (25 test projects per group), then fine-tunes
one CodeBERT model per group. Each epoch runs one training pass plus two full evaluation passes
(training set and validation set), so an epoch costs roughly 1.7x a plain training pass.

Output is verbose; the useful signal is the per-epoch `EarlyStopping` line reporting validation
macro-F1.

In [ ]:
import os; os.chdir(SRC); print("cwd:", os.getcwd())
!python -W ignore Bert_train_per_project.py "{REPO_DIR}/{DATASET_REL}" "{WEIGHTS_PREFIX_REL}" "{DATA_DIR_NAME}" "{TECHNIQUE}"

## 8. Inspect the checkpoints

Checkpoints are about 500 MB each. Everything under `/kaggle/working` is saved as notebook output,
which is capped at 20 GB - enough for the full four-group run, but worth watching if you re-run
without cleaning up.

In [ ]:
import glob, os

for f in sorted(glob.glob(f"{REPO_DIR}/models/*.pt")):
    print(f"{os.path.getsize(f)/1e6:8.1f} MB  {f}")

print()
print("splits written to:", f"{SRC}/{DATA_DIR_NAME}")
for f in sorted(glob.glob(f"{SRC}/{DATA_DIR_NAME}/*.csv")):
    print("  ", os.path.basename(f))

## 9. Evaluate

Loads `..._project_group_N.pt` for each group and prints per-category precision, recall and F1.

In [ ]:
import os; os.chdir(SRC); print("cwd:", os.getcwd())
!python -W ignore Testing_per_project.py "{REPO_DIR}/{DATASET_REL}" "{WEIGHTS_PREFIX_REL}" "calculate_attribution_False" "{DATA_DIR_NAME}" "{TECHNIQUE}"

In [ ]:
# The evaluation script derives its summary filename from the part of the technique tag before the
# first "-", so results land in results/per_Category_Evaluation_BERT.txt rather than the name used
# by the repository's rq1.sh parser. Print whatever was produced.
import glob

for f in sorted(glob.glob(f"{REPO_DIR}/results/per_Category_Evaluation_*.txt")):
    print("==", f)
    print(open(f).read())

for f in sorted(glob.glob(f"{SRC}/*-result/*.csv")):
    print("predictions:", f)

## Notes and known pitfalls

**Do not subset the projects to make the run smaller.** `create_train_test_groups` in
`src/data_processing.py` reshuffles in a `while True` loop until every test group contains at least
four examples of each category 0-5. The rarest categories have only 33-41 examples in the entire
dataset, so a reduced project list can make that condition unsatisfiable and the loop will spin
forever with no error. Reduce epochs and group count instead, as this notebook does.

**Splits are not reproducible.** `random.shuffle` in `create_train_test_groups` is unseeded, so each
run draws different test projects. Seed it before comparing runs against each other.

**Class imbalance.** 8,294 of 8,574 examples are non-flaky. Macro-F1 on the rare categories will look
poor after three epochs - that is expected for a smoke run, not a defect. The training code already
compensates with balanced class weights and focal loss.

**TensorBoard graph logging.** `writer.add_graph` traces the chunked forward pass and can raise on
some torch versions. If it does, comment out that line in `src/Bert_train_per_project.py` - it only
affects logging.

**Session limits.** A Kaggle session runs for at most 12 hours, and GPU quota is limited per week.
The full four-group, 40-epoch run takes 8-10 hours, so it fits one session but leaves little margin.
Consider running one group per session and keeping the checkpoints as notebook outputs.

**Scaling up.** Set `GROUPS = 4` and `EPOCHS = 40` in the Configuration cell, then re-run from the
patch cell onward.